In [3]:
import pandas as pd
import numpy as np

In [4]:
df_house = pd.read_csv('bengaluru_house_prices.csv')

In [5]:
df_house = df_house.drop(['area_type', 'society', 'balcony', 'availability'], axis=1)

In [6]:
df_house['location'] = df_house['location'].fillna('Whitefield')
df_house['size'] = df_house['size'].fillna('2 BHK')
df_house['bath'] = df_house['bath'].fillna(df_house['bath'].median())

In [7]:
df_house['bhk'] = df_house['size'].apply(lambda x: int(x.split(' ')[0]))

In [8]:
def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df_house['total_sqft'] = df_house['total_sqft'].apply(convert_sqft_to_num)
df_house = df_house.dropna()

In [9]:
df_house.location = df_house.location.apply(lambda x: x.strip())
location_stats = df_house.groupby('location')['location'].agg('count')
locations_less_than_10 = location_stats[location_stats <= 10]
df_house.location = df_house.location.apply(lambda x: 'other' if x in locations_less_than_10 else x)

In [10]:
dummies = pd.get_dummies(df_house.location, drop_first=True)
df_house_final = pd.concat([df_house.drop(['location', 'size'], axis=1), dummies], axis=1)

In [11]:
X_house = df_house_final.drop('price', axis=1)
y_house = df_house_final['price']

In [13]:
from sklearn.model_selection import GridSearchCV, ShuffleSplit
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor

In [14]:
def find_best_house_model(X, y):
    algos = {
        'linear_regression': {
            'model': LinearRegression(),
            'params': {}
        },
        'lasso': {
            'model': Lasso(),
            'params': {
                'alpha': [1, 2],
                'selection': ['random', 'cyclic']
            }
        },
        'decision_tree': {
            'model': DecisionTreeRegressor(),
            'params': {
                'criterion': ['squared_error', 'friedman_mse'],
                'splitter': ['best', 'random']
            }
        }
    }
    scores = []
    cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
    
    for algo_name, config in algos.items():
        gs = GridSearchCV(config['model'], config['params'], cv=cv, return_train_score=False)
        gs.fit(X, y)
        scores.append({
            'model': algo_name,
            'best_score': gs.best_score_,
            'best_params': gs.best_params_
        })

    return pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])